# 01 - Exploratory Data Analysis (EDA)

This notebook explores the churn dataset and engineered features used by the production pipeline.

**Prerequisites**:
- Run the ingestion and feature pipelines first, so `features_train.csv` exists:
  - `make ingest`
  - `make features`

We will:
- Inspect the shape and target distribution.
- Look at key feature distributions and relationships.
- Build a bit of intuition for churn drivers that align with the production model.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import EXPORTS_DIR

sns.set_theme(style="whitegrid")

print(f"Using exports directory: {EXPORTS_DIR}")

In [ ]:
# Load training features
features_path = EXPORTS_DIR / "features_train.csv"
features_df = pd.read_csv(features_path)

print(f"Loaded features_train.csv with shape: {features_df.shape}")
features_df.head()

In [ ]:
# Target distribution
churn_rate = features_df["is_churn"].mean()
print(f"Overall churn rate: {churn_rate:.2%}")

fig, ax = plt.subplots(figsize=(4, 4))
features_df["is_churn"].value_counts(normalize=True).rename({0: "No churn", 1: "Churn"}).plot.bar(ax=ax)
ax.set_ylabel("Proportion")
ax.set_title("Churn vs Non-churn")
plt.tight_layout()

In [ ]:
# Example feature distributions for a few key drivers
num_features = [
    "usage_decay_30d",
    "sessions_last_30d",
    "login_gap_days_avg",
    "ticket_count_90d",
    "payment_failure_count_90d",
]

available = [c for c in num_features if c in features_df.columns]
features_df[available].describe().T

In [ ]:
# Churn vs non-churn for selected numeric features
import numpy as np

if available:
    fig, axes = plt.subplots(nrows=1, ncols=min(3, len(available)), figsize=(15, 4))
    if not isinstance(axes, (list, np.ndarray)):
        axes = [axes]

    for ax, col in zip(axes, available[:3]):
        sns.kdeplot(
            data=features_df,
            x=col,
            hue="is_churn",
            common_norm=False,
            fill=True,
            alpha=0.4,
            ax=ax,
        )
        ax.set_title(col)

    plt.tight_layout()
else:
    print("No numeric features from the selected list were found in features_train.csv")

In [ ]:
# Relationship between risk_tier (if available) and churn

if "risk_tier" in features_df.columns:
    fig, ax = plt.subplots(figsize=(6, 4))
    tier_churn = (
        features_df.groupby("risk_tier")["is_churn"].mean().sort_values(ascending=False)
    )
    tier_churn.plot.bar(ax=ax)
    ax.set_ylabel("Churn rate")
    ax.set_title("Churn rate by risk_tier")
    plt.tight_layout()
else:
    print("Column 'risk_tier' not found in features_train.csv")